# Lesson 7 : Agent Skills

Agent Skill is first introduced by Anthropic on October 2025 and now it's an open (and cross-platform) project. (See [here](https://agentskills.io/home) for Agent Skill's specification.)  
The objective of Agent Skills is to enable highly modularization and reuse of Agent skills and knowledge - such as, instructions, tool usage, and knowledge retrieval.

In this exercise, we create a brief custom skill to create a expense report, and we then use this skill with Agent Framework.

## Create a custom file skill

There exist a lot of pre-built (reusable) existing skills (see [here](https://github.com/heilcheng/awesome-agent-skills)), but in this exercise, we briefly build a custom file skill to create a unique expense report for our virtual company as follows.

First we create a file (named ```SKILL.md```) to describe skill's body.  
The function for calculating the additional fee will be defined later as a separate skill.

In [1]:
import os
skill_folder = path = os.path.join("skills", "corporate-expense-report")
os.makedirs(skill_folder, exist_ok=True)

In [2]:
%%writefile skills/corporate-expense-report/SKILL.md
---
name: corporate-expense-report
description: This skill calculates additional fee in each expense item and create a corporate expense report using template.
---

# Corporate Expense Report Skill

This skill provides instructions on how to create a corporate expense report.

## Capabilities

- Infer the category from the description in each expense item.  
  Available categories: "international transport", "domestic transport", "meal", and "misc"
- Calculate additional fee (tax + transaction fee) for each item
- Generate a expense report including additional fee

## Input Format

Provide the list of expense - in which each item includes description, date, and amount.

## Output Format

Show expense report using template: [assets/expense_template.md](assets/expense_template.md)

Writing skills/corporate-expense-report/SKILL.md


In above skill's body, we refer an asset ```expense_template.md``` which includes output template for expense report.  
Now we create this asset (```expense_template.md```) as follows.

In [3]:
asset_folder = path = os.path.join(skill_folder, "assets")
os.makedirs(asset_folder, exist_ok=True)

In [4]:
%%writefile skills/corporate-expense-report/assets/expense_template.md
| Date | Category | Amount (USD) | Additional fee (USD) | Sub total (USD) |
|------|----------|--------------|----------------------|-----------------|
|      |          |              |                      |                 |

Total: [total fee]

Writing skills/corporate-expense-report/assets/expense_template.md


The ```skills``` folder will then have the following structure.  
In this exercise, we use only a single file skill to solve the problem, but you can include a lot of existing skills in ```skills``` folder.

```
skills/
└── corporate-expense-report/
    ├── SKILL.md
    └── assets/
        └── expense_template.md
```

## Create a code skill

Sometimes we need a resource to execute logic code at read time, rather than static text.  
In such cases, we have been using tools running in-process as direct function calls (such as, local function tools) or code interpreter tools in previous exmaples, but in skill's framework, we can use code-defined skills to address these scenarios.

In this example, we define a code skill as follows, in which it can calculate additional fees (tax + transaction fee) according to our own corporate policy.

In [5]:
from agent_framework import Skill

expense_tasks_skill = Skill(
    name="expense-tasks",
    description="Additional expense operations",
    content="Use this skill when additional tasks to do is required in expense.",
)

@expense_tasks_skill.script(
    name="additional-fee-calculator",
    description=(
        "Calculate additional fee (total of tax and transaction fee) of expense item according to corporate policy. "
        "The ```category``` argument should be one of  - ```international transport```, ```domestic transport```, ```meal```, or ```misc```"
    )
)
def calculate_additional_fee(category: str, amount: int) -> int:
    """Calculate additional fee (tax + transaction) for the expense item.

    Args:
        category:       One of items - "international transport", "domestic transport", "meal", or "misc"
        amount:         The amount of this category

    Returns:
        Additional fee (Tax + transaction fee)
    """
    tax_ratio = 0.0
    transaction_ratio = 0.0

    if category.lower() == "international transport":
        tax_ratio = 0.00
        transaction_ratio = 0.10
    elif category.lower() == "domestic transport":
        tax_ratio = 0.00
        transaction_ratio = 0.05
    elif category.lower() == "meal":
        tax_ratio = 0.10
        transaction_ratio = 0.00
    elif category.lower() == "misc":
        tax_ratio = 0.03
        transaction_ratio = 0.00
    else:
        raise Exception(f"unsupported category: {category.lower()}")

    return int(amount * tax_ratio) + int(amount * transaction_ratio)

## Run agent with skills

Now let's build an agent to use skills.

First we initilize the client object as usual.

In [6]:
from dotenv import load_dotenv
from agent_framework.azure import AzureAIClient
from azure.identity.aio import AzureCliCredential

load_dotenv()

credential = AzureCliCredential()
client = AzureAIClient(
    credential=credential,
)

Now we define a skill provider, which refers to above skill folder and a code-based custom skill.

In [7]:
from agent_framework import SkillsProvider

skills_provider = SkillsProvider(
    skill_paths="skills",
    skills=[expense_tasks_skill]
)

Now we create an agent with this skill provider as follows.  
Same as Lesson 6, we set a provider in ```context_providers``` property in the agent.

In [8]:
from agent_framework import Agent

agent = Agent(
    name="AgentWithSkills",
    client=client,
    instructions="You are a helpful assistant that can write and execute Python code to solve problems.",
    context_providers=[skills_provider],
)

Now let's run the agent.  
In this skill, the following ration of tax and transaction fee is added in each item, and the agent shows an expense report with a table format which columns are "data", "category", "amount", "tax and transaction fee", and "sub total".

| category | tax ratio | transaction fee ratio |
|----------|-----------|-----------------------|
| international transport | 0.00 | 0.10 |
| domestic transport | 0.00 | 0.05 |
| meal | 0.10 | 0.00 |
| misc | 0.03 | 0.00 |

In [9]:
from IPython.display import Markdown, display

prompt = """Return an expense report of the following.

- flight from JFK to SEA (round trip)
    - date : 2026/02/09
    - amount : 2800
- dinner
    - date : 2026/02/09
    - amount : 80
- lunch
    - date : 2026/02/10
    - amount : 37
- souvenirs
    - date : 2026/02/10
    - amount : 40
"""

result = await agent.run(prompt)
display(Markdown(result.text))

| Date | Category | Amount (USD) | Additional fee (USD) | Sub total (USD) |
|------|----------|--------------|----------------------|-----------------|
| 2026/02/09 | domestic transport | 2800.00 | 140.00 | 2940.00 |
| 2026/02/09 | meal | 80.00 | 8.00 | 88.00 |
| 2026/02/10 | meal | 37.00 | 3.00 | 40.00 |
| 2026/02/10 | misc | 40.00 | 1.00 | 41.00 |

Total: 3109.00 USD